# Handling Tables in a PDF

____________________________________________

### 🔹 Lets's work with US Regulations Document  
Preview

<img src="assets/us_regulations_pdf_side_by_side.png" height="300">

##  Loading PDF with pdfplumber

In [3]:
import pdfplumber

pdf_path = "data-reports/us_regulations.pdf"

plumber_doc = pdfplumber.open(pdf_path)
print(f"Length of the document: {len(plumber_doc.pages)}")

Length of the document: 2


## find_tables() built-in fuction finds the tables  preserves the structure

In [8]:
for page_num, page in enumerate(plumber_doc.pages):
    tables = page.find_tables()
    print(f"Page {page_num+1} -> {len(tables)} tables detected")

Page 1 -> 2 tables detected
Page 2 -> 1 tables detected


## Let's visualize the raw structure of extracted table 

In [ ]:
with pdfplumber.open(pdf_path) as pdf:
    for page_num, page in enumerate(pdf.pages):
        tables = page.find_tables()
        
        print(f"\n========== PAGE {page_num+1} ==========")
        
        for i, table in enumerate(tables):
            raw_table = table.extract()
            
            print(f"\n--- Raw Table {i+1} Structure ---")
            for row in raw_table:
                print(row)


========== PAGE 1 ==========

--- Raw Table 1 Structure ---
['Table 1.1 – Water Exposure Test Limits', None]
['Part Code', 'Vibration Limit (mm/s)']
['L102', '3.0']
['M330', '4.5']
['X778', '4.0']
['A450', '3.0']
['B990', '3.5']

--- Raw Table 2 Structure ---
['Table 1.2 – Dust Exposure Test Limits', None]
['Part Code', 'Vibration Limit (mm/s)']
['L102', '5.0']
['M330', '4.5']
['X778', '2.5']
['A450', '3.0']
['B990', '5.0']

========== PAGE 2 ==========

--- Raw Table 1 Structure ---
['Table 1.3 – Mud Exposure Test Limits', None]
['Part Code', 'Vibration Limit (mm/s)']
['L102', '3.0']
['M330', '2.0']
['X778', '4.0']
['A450', '2.0']
['B990', '4.0']


## Function to convert the table to markdown 
- LLM friendly and easier to do further transformations if needed.

In [16]:
def table_to_markdown(table):
    if not table:
        return ""
    
    headers = table[0]
    rows = table[1:]
    
    md = "| " + " | ".join(str(h or "") for h in headers) + " |\n"
    md += "| " + " | ".join(["---"] * len(headers)) + " |\n"
    
    for row in rows:
        md += "| " + " | ".join(str(c or "") for c in row) + " |\n"
    
    return md

In [19]:
all_tables = []

for page_num, page in enumerate(plumber_doc.pages):
    for table in page.find_tables():
        md = table_to_markdown(table.extract())
        
        if md:
            all_tables.append({
                "type": "table",
                "content": md,
                "page": page_num + 1,
                "y": table.bbox[1]
            })

for t in all_tables:
    print("\n--- TABLE ---")
    print(t["content"])



--- TABLE ---
| Table 1.1 – Water Exposure Test Limits |  |
| --- | --- |
| Part Code | Vibration Limit (mm/s) |
| L102 | 3.0 |
| M330 | 4.5 |
| X778 | 4.0 |
| A450 | 3.0 |
| B990 | 3.5 |


--- TABLE ---
| Table 1.2 – Dust Exposure Test Limits |  |
| --- | --- |
| Part Code | Vibration Limit (mm/s) |
| L102 | 5.0 |
| M330 | 4.5 |
| X778 | 2.5 |
| A450 | 3.0 |
| B990 | 5.0 |


--- TABLE ---
| Table 1.3 – Mud Exposure Test Limits |  |
| --- | --- |
| Part Code | Vibration Limit (mm/s) |
| L102 | 3.0 |
| M330 | 2.0 |
| X778 | 4.0 |
| A450 | 2.0 |
| B990 | 4.0 |

